In [1]:
import os
import sys
print(os.getcwd())
sys.path.append('/root/data1/Code/Mumu/FlagEmbedding/FlagEmbedding/visual/VISTA_Evaluation_FineTuning-main/evaluation_example_webqa/BGE-base')
import faiss
import torch
import logging
import datasets
import numpy as np
from tqdm import tqdm
from typing import Optional
from dataclasses import dataclass, field
from transformers import HfArgumentParser
# from FlagEmbedding import FlagModel
from flag_eva_token_new import Flag_bgev_model
# from flag_clip import Flag_clip
import json

/root/data1/Code/Mumu/FlagEmbedding/FlagEmbedding/visual/VISTA_Evaluation_FineTuning-main/evaluation_example_webqa/BGE-base


/conda/envs/kgcopy/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/conda/envs/kgcopy/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
def append_to_json_file(data, file_path):
    # 检查文件是否存在
    if os.path.exists(file_path):
        # 如果文件存在，读取当前的JSON数据
        with open(file_path, "r") as f:
            try:
                file_data = json.load(f)  # 读取已有的JSON文件内容
            except json.JSONDecodeError:
                file_data = []  # 如果文件是空的或者格式错误，初始化为空列表
    else:
        file_data = []  # 文件不存在时，初始化为空列表

    # 将新数据追加到文件数据中
    file_data.append(data)

    # 将更新后的数据重新写入JSON文件
    with open(file_path, "w") as f:
        json.dump(file_data, f, indent=4)

In [3]:
resume_path = "/root/data1/Code/Mumu/save_models_caption/checkpoint-2900/BGE_EVA_Token.pth"
image_dir = "/root/data5/MuMuQA/resize_images263000/"
model = Flag_bgev_model(model_name_bge = "BAAI/bge-base-en-v1.5",
                    model_name_eva = "EVA02-CLIP-B-16", # "EVA02-CLIP-B-16",
                    normlized = True,
                    eva_pretrained_path = "eva_clip",
                    resume_path=resume_path,
                    image_dir=image_dir,
                    )

/root/data1/Code/Mumu/FlagEmbedding/FlagEmbedding/visual/eva_clip/factory.py:87: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_l

True
cuda
----------using 2*GPUs----------


In [4]:
eval_data = datasets.load_dataset('json', data_files="/root/data1/Code/Mumu/processdata/retrive_goundtruth_ansnoNone.json", split='train')
text_corpus = datasets.load_dataset('json', data_files="/root/data1/Code/Mumu/processdata/retrive_goundtruth_ansnoNone.json", split='train')

In [5]:
def search(model: Flag_bgev_model, queryQ,queryI, corpus,k:int = 100, batch_size: int = 1, max_length: int=512):
    """
    1. Encode queries into dense embeddings;
    2. Search through faiss index
    """
    # model.eval()
    with torch.no_grad():
        query_embedding = model.encode_queries([queryQ], batch_size=batch_size, max_length=max_length, query_type="text")
        text_corpus = corpus
        text_corpus_embeddings = model.encode_corpus_item(text_corpus, batch_size=batch_size, max_length=max_length, corpus_type='text')
        similarity = query_embedding @ text_corpus_embeddings.T
        similarity = similarity.squeeze()
        top_indices = np.argsort(-similarity)[:k]
        
        top_sentences = [text_corpus[i] for i in top_indices]

        return top_sentences

In [6]:
print(len(eval_data))
for idx,item in enumerate(tqdm(list(eval_data)[:])):
    extend = item["image"].split('.')[-1]
    pic_id = item["voa_image_id"]+ '.'+extend
    # ques = "Caption : "+ item["caption"] +"Question : " + id2ques[item["data_id"]]
    ques = "Question : " + item["question"]
    top_sentences = search(
            model=model, 
            queryQ=ques, 
            queryI = pic_id,
            corpus = item['text'],
            k=100,
            batch_size=1, 
            max_length=512
        )
    data = {
            'data_id': item["id"], 
            'question': ques, 
            'caption': item["caption"],
            'candidates': item["candidates"],
            'retrieved': top_sentences,
        }
    append_to_json_file(data,'/root/data1/Code/Mumu/data/modality/text/textRetrieval.json') 

1121


100%|██████████████████████████████████████████████| 1121/1121 [09:15<00:00,  2.02it/s]
